In [6]:
import pandas as pd
import re
import unicodedata

df = pd.read_csv(
    "amazon_kozmetik_data.csv",
    encoding="utf-8-sig"
)

In [7]:
df.head()

,asin,title,price,old_price,rating,review_count,sales_last_month,link
0,B0BJ7F5XJS,Golden Rose Longstay Liquid Matte Lipstick No:...,355.00,355.00,NaN,NaN,NaN,NaN
1,B0DN29JGN2,Note Cosmetics 3 in 1 Healthy Skin Tinted Mois...,284.90,284.90,3.9,32.0,50.0,NaN
2,B0CB8KW5PH,Note Cosmetics Collagen Concealer 01 Kolajen İ...,350.95,350.95,4.0,28.0,NaN,NaN
3,B0855963RQ,Note Cosmetics Mineral Concealer 202 SPF 15 Gö...,254.90,254.90,4.4,74.0,NaN,NaN
4,B0078MF1PC,L'Oreal Paris Telescopic Maskara - Siyah,499.00,499.00,4.3,13.0,500.0,NaN


In [8]:
df.shape

(251, 8)

In [9]:
df.columns

Index(['asin', 'title', 'price', 'old_price', 'rating', 'review_count',
       'sales_last_month', 'link'],
      dtype='object')

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251 entries, 0 to 250
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   asin              251 non-null    object 
 1   title             251 non-null    object 
 2   price             251 non-null    float64
 3   old_price         167 non-null    float64
 4   rating            204 non-null    float64
 5   review_count      204 non-null    float64
 6   sales_last_month  43 non-null     float64
 7   link              0 non-null      float64
dtypes: float64(6), object(2)
memory usage: 15.8+ KB


In [11]:
print("\nEksik değerler:")
print(df.isnull().sum())


Eksik değerler:
asin                  0
title                 0
price                 0
old_price            84
rating               47
review_count         47
sales_last_month    208
link                251
dtype: int64


In [12]:
print("\nTekrarlı kayıt sayısı:")
print(df.duplicated().sum())


Tekrarlı kayıt sayısı:
0


In [13]:
df = df.drop(
    columns=[
        "asin",
        "link"
    ],
    errors="ignore"
)

print(df.head())
print(df.columns)

                                               title   price  old_price  \
0  Golden Rose Longstay Liquid Matte Lipstick No:...  355.00     355.00   
1  Note Cosmetics 3 in 1 Healthy Skin Tinted Mois...  284.90     284.90   
2  Note Cosmetics Collagen Concealer 01 Kolajen İ...  350.95     350.95   
3  Note Cosmetics Mineral Concealer 202 SPF 15 Gö...  254.90     254.90   
4           L'Oreal Paris Telescopic Maskara - Siyah  499.00     499.00   

   rating  review_count  sales_last_month  
0     NaN           NaN               NaN  
1     3.9          32.0              50.0  
2     4.0          28.0               NaN  
3     4.4          74.0               NaN  
4     4.3          13.0             500.0  
Index(['title', 'price', 'old_price', 'rating', 'review_count',
       'sales_last_month'],
      dtype='object')


In [14]:
old_price_mean = df["old_price"].mean()

print("Old Price Ortalama:", old_price_mean)

df["old_price"] = np.where(
    df["old_price"].isna(),
    np.maximum(old_price_mean, df["price"]),
    df["old_price"]
)

Old Price Ortalama: 485.8617964071857


In [15]:
same_price_count = (
    df["price"] == df["old_price"]
).sum()

print(
    "Price = Old Price olan kayıt:",
    same_price_count
)

Price = Old Price olan kayıt: 126


In [16]:
df["discount_rate"] = (
    (df["old_price"] - df["price"])
    / df["old_price"]
) * 100

In [17]:
print(df.isnull().sum())

title                 0
price                 0
old_price             0
rating               47
review_count         47
sales_last_month    208
discount_rate         0
dtype: int64


In [18]:
rating_mean = df["rating"].mean()

review_mean = df["review_count"].mean()

print("Rating Ortalama:", rating_mean)
print("Review Ortalama:", review_mean)

df["rating"] = df["rating"].fillna(rating_mean)

df["review_count"] = df["review_count"].fillna(review_mean)

Rating Ortalama: 4.258333333333333
Review Ortalama: 102.33823529411765


In [19]:
print(df[["rating", "review_count"]].isnull().sum())

rating          0
review_count    0
dtype: int64


In [20]:
mask = df["sales_last_month"].isna()

df.loc[mask, "sales_last_month"] = np.random.randint(
    50,
    501,
    size=mask.sum()
)

In [21]:
print(df["sales_last_month"].isnull().sum())

0


In [22]:
unique_titles = df["title"].dropna().unique()

for i, title in enumerate(unique_titles, start=1):
    print(f"{i}. {title}")

print("\n" + "="*50)
print("Toplam eşsiz title sayısı:", len(unique_titles))

1. Golden Rose Longstay Liquid Matte Lipstick No: 42 - Bulaşmayan Kalıcı Likit Mat Ruj
2. Note Cosmetics 3 in 1 Healthy Skin Tinted Moisturizer Renk Ton Eşitleyici 50 SPF Aydınlatıcı Krem
3. Note Cosmetics Collagen Concealer 01 Kolajen İçerikli SPF 20 Göz Altı Kapatıcısı
4. Note Cosmetics Mineral Concealer 202 SPF 15 Göz Altı Kapatıcısı
5. L'Oreal Paris Telescopic Maskara - Siyah
6. Blistex Medplus Stick Kuru ve Çatlamış Dudaklara Onarıcı ve Ferahlatıcı Dudak Bakım Kremi SPF 15
7. Cream Co. Su Bazlı Moisturizer Nemlendirici Aydınlatıcı Yüz Kremi, Hyaluronik Asit, 50 ML, Tüm Cilt Tipleri
8. The Purest Solutions, Gözenek ve Siyah Nokta Görünümünü Azaltmaya Yardımcı Arındırıcı Tonik,%5 Glikolik Asit, AHA, BHA, 200 ml
9. L'Oreal Paris Elseve Mucizevi Yağ Saç Güzelleştirici Krem 150 ml - Kuru ve Sert Saçlar İçin
10. L'Oreal Paris Lumi Blush Likit Allık - 630 Glowy True Rose
11. The Purest Solutions, Retinol İçerikli Gece Bakım Serumu, Kırışıklık Karşıtı Bakım, 30 ml (%1 Retinol + Seramid)
1

In [23]:
brand_map = {
    "maybelline": "Maybelline",
    "loreal": "L'Oréal",
    "loreal paris": "L'Oréal",
    "nivea": "Nivea",
    "NIVEA":"nivea",
    "farmasi": "Farmasi",
    "garnier": "Garnier",
    "flormar": "Flormar",
    "pastel": "Pastel",
    "golden rose": "Golden Rose",
    "note cosmetics": "Note Cosmetics",
    "note": "Note Cosmetics",
    "cream co": "Cream Co",
    "la roche posay": "La Roche-Posay",
    "urban care": "URBAN Care",
    "dove": "Dove",
    "neutrogena": "Neutrogena",
    "blistex": "Blistex",
    "the purest solutions": "The Purest Solutions",
    "cerave": "CeraVe",
    "missha": "MISSHA",
    "doa kozmetik": "Doa Kozmetik",
    "clasy care": "Clasy Care",
    "clasycare": "Clasy Care",
    "vichy": "Vichy",
    "vaseline": "Vaseline",
    "gabrini": "Gabrini",
    "nascita": "Nascita",
    "pierre cardin": "Pierre Cardin",
    "bee beauty": "Bee Beauty",
    "doa": "Doa Kozmetik"
}

In [24]:
category_map = {
  
    "ruj": "Makyaj",
    "lipstick": "Makyaj",
    "lipgloss": "Makyaj",
    "gloss": "Makyaj",
    "maskara": "Makyaj",
    "mascara": "Makyaj",
    "eyeliner": "Makyaj",
    "eyeshadow": "Makyaj",
    "allık": "Makyaj",
    "blush": "Makyaj",
    "concealer": "Makyaj",
    "fondöten": "Makyaj",
    "foundation": "Makyaj",
    "brow": "Makyaj",
    "pencil": "Makyaj",
    
    # SKINCARE
    "serum": "Cilt Bakımı",
    "cream": "Cilt Bakımı",
    "moisturizer": "Cilt Bakımı",
    "tonik": "Cilt Bakımı",
    "cleanser": "Cilt Bakımı",
    "mask": "Cilt Bakımı",
    "spf": "Cilt Bakımı",
    "anti aging": "Cilt Bakımı",
    "retinol": "Cilt Bakımı",
    "niacinamide": "Cilt Bakımı",
    "vitamin c": "Cilt Bakımı",
    
    # HAIR CARE
    "shampoo": "Saç Bakım",
    "saç": "Saç Bakım",
    "hair": "Saç Bakım",
    
    # PERFUME
    "parfüm": "Parfüm",
    "edp": "Parfüm",
    "edt": "Parfüm",
    
    # ACCESSORIES
    "çanta": "Aksesuar",
    "organizer": "Aksesuar",
    "fırça": "Aksesuar",
    "brush": "Aksesuar",
    "makyaj çantası": "Aksesuar",
    "set": "Aksesuar",
    "kit": "Aksesuar",
    "bottle": "Aksesuar",
}

In [25]:
def normalize(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_brand(text):
    text = normalize(text)
    
    for k, v in sorted(brand_map.items(), key=lambda x: -len(x[0])):
        if k in text:
            return v
    return np.nan


def extract_category(text):
    text = normalize(text)
    
    for k, v in category_map.items():
        if k in text:
            return v
    return "Other"


df["clean_title"] = df["title"].apply(normalize)
df["brand"] = df["title"].apply(extract_brand)
df["category"] = df["title"].apply(extract_category)

In [26]:
df[["brand", "category"]].isna().sum()

brand       125
category      0
dtype: int64

In [27]:
df.to_csv("kozmetik_dataset_preprocessing_1.csv", index=False, encoding="utf-8-sig")

In [28]:
df.to_excel("kozmetik_dataset_duzenlenmis.xlsx", index=False, engine="openpyxl")

In [1]:
df = pd.read_excel(
    "kozmetik_dataset_preprocessing_2.xlsx")
print("done")

done


In [2]:
df = df.drop(columns=["brand", "title", "category_extracted"], errors="ignore")

In [3]:
df.columns

Index(['price', 'old_price', 'rating', 'review_count', 'sales_last_month',
       'discount_rate', 'clean_title', 'brand_extracted', 'category'],
      dtype='object')

In [4]:
df.isna().sum()

price               0
old_price           0
rating              0
review_count        0
sales_last_month    0
discount_rate       0
clean_title         0
brand_extracted     0
category            0
dtype: int64

In [5]:
df.to_csv("kozmetik_dataset_clean_1.csv", index=False)

In [6]:
df = pd.read_csv(
    "kozmetik_dataset_clean_1.csv",
    encoding="utf-8-sig"
)

In [8]:
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["old_price"] = pd.to_numeric(df["old_price"], errors="coerce")

print("Price ortalama:", df["price"].mean())
print("Old Price ortalama:", df["old_price"].mean())

def detect_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    
    return outliers, lower_bound, upper_bound

price_outliers, price_low, price_high = detect_outliers(df, "price")
print("\nPRICE outlier sayısı:", len(price_outliers))
print("PRICE sınırlar:", price_low, price_high)

old_price_outliers, old_low, old_high = detect_outliers(df, "old_price")
print("\nOLD_PRICE outlier sayısı:", len(old_price_outliers))
print("OLD_PRICE sınırlar:", old_low, old_high)

outliers_all = pd.concat([price_outliers, old_price_outliers]).drop_duplicates()

Price ortalama: 1613.0987550200803
Old Price ortalama: 550.0083887646394

PRICE outlier sayısı: 29
PRICE sınırlar: -644.9499999999998 1650.81

OLD_PRICE outlier sayısı: 35
OLD_PRICE sınırlar: -78.80999999999995 847.27


In [9]:
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["old_price"] = pd.to_numeric(df["old_price"], errors="coerce")

price_mean = df["price"].mean()
old_price_mean = df["old_price"].mean()

def get_bounds(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return lower, upper

price_low, price_high = get_bounds(df, "price")
old_low, old_high = get_bounds(df, "old_price")

df.loc[(df["price"] < price_low) | (df["price"] > price_high), "price"] = price_mean

df.loc[(df["old_price"] < old_low) | (df["old_price"] > old_high), "old_price"] = old_price_mean

df.loc[df["old_price"] > df["price"], "old_price"] = df["price"]

print("Price mean:", df["price"].mean())
print("Old Price mean:", df["old_price"].mean())

Price mean: 572.5011803035435
Old Price mean: 342.06064277792706


In [30]:
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["old_price"] = pd.to_numeric(df["old_price"], errors="coerce")

df["discount_rate"] = ((df["old_price"] - df["price"]) / df["old_price"]) * 100

df["discount_rate"] = df["discount_rate"].replace([np.inf, -np.inf], np.nan).fillna(0)

df["discount_rate"] = df["discount_rate"].round(2)

print(df[["price", "old_price", "discount_rate"]].head())

    price  old_price  discount_rate
0  355.00     355.00            0.0
1  284.90     284.90            0.0
2  350.95     350.95            0.0
3  254.90     254.90            0.0
4  499.00     499.00            0.0


In [31]:
df["price"] = df["price"].round(2)
df["old_price"] = df["old_price"].round(2)

In [32]:
df.head()

,price,old_price,rating,review_count,sales_last_month,discount_rate,clean_title,brand_extracted,category
0,355.00,355.00,4.3,102,287,0.0,golden rose longstay liquid matte lipstick no ...,Golden Rose,Makyaj
1,284.90,284.90,3.9,32,50,0.0,note cosmetics 3 in 1 healthy skin tinted mois...,Note Cosmetics,Cilt Bakımı
2,350.95,350.95,4.0,28,425,0.0,note cosmetics collagen concealer 01 kolajen i...,Note Cosmetics,Makyaj
3,254.90,254.90,4.4,74,80,0.0,note cosmetics mineral concealer 202 spf 15 go...,Note Cosmetics,Makyaj
4,499.00,499.00,4.3,13,500,0.0,l oreal paris telescopic maskara siyah,Lore,Makyaj


In [33]:
print(df[["price", "old_price"]].describe())

             price   old_price
count   249.000000  249.000000
mean    572.501325  342.060843
std     491.854501  173.439887
min      21.500000   21.500000
25%     215.960000  199.000000
50%     399.000000  340.000000
75%     789.900000  499.000000
max    1613.100000  789.900000


In [34]:
unique_ratings = df["rating"].unique()
print(unique_ratings)

[4.3 3.9 4.  4.4 4.7 4.5 4.6 4.2 5.  4.1 3.4 3.5 3.6 3.8 2.6 2.4 2.8 3.2
 2.  4.9 4.8 1.  3.  3.3 2.9 3.7]


In [35]:
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

df["rating"] = df["rating"].round(1)

print(df["rating"].unique())

[4.3 3.9 4.  4.4 4.7 4.5 4.6 4.2 5.  4.1 3.4 3.5 3.6 3.8 2.6 2.4 2.8 3.2
 2.  4.9 4.8 1.  3.  3.3 2.9 3.7]


In [36]:
df["review_count"] = pd.to_numeric(df["review_count"], errors="coerce")

df["review_count"] = df["review_count"].astype(int)

print(df["review_count"].describe())

count    249.000000
mean     102.277108
std      192.896365
min        1.000000
25%        3.000000
50%       32.000000
75%      102.000000
max      972.000000
Name: review_count, dtype: float64


In [37]:
df["review_count"] = df["review_count"].round(2)

In [38]:
df.to_csv("kozmetik_dataset_preprocessing_end.csv", index=False)